Here we Load our FAISS index + model.
Expose a simple chat interface in browser.
Keep history so we can ask follow-ups.

In [1]:
!pip install gradio

   ---------------------------------------- 0.0/59.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/59.6 MB ? eta -:--:--
    --------------------------------------- 1.0/59.6 MB 3.1 MB/s eta 0:00:19
   - -------------------------------------- 1.8/59.6 MB 3.3 MB/s eta 0:00:18
   - -------------------------------------- 2.4/59.6 MB 3.3 MB/s eta 0:00:18
   - -------------------------------------- 2.9/59.6 MB 3.3 MB/s eta 0:00:18
   -- ------------------------------------- 3.4/59.6 MB 2.9 MB/s eta 0:00:20
   -- ------------------------------------- 3.9/59.6 MB 2.9 MB/s eta 0:00:20
   -- ------------------------------------- 4.5/59.6 MB 2.8 MB/s eta 0:00:20
   --- ------------------------------------ 4.7/59.6 MB 2.7 MB/s eta 0:00:21
   --- ------------------------------------ 5.5/59.6 MB 2.7 MB/s eta 0:00:20
   ---- ----------------------------------- 6.0/59.6 MB 2.8 MB/s eta 0:00:20
   ---- ----------------------------------- 6.3/59.6 MB 2.7 MB/s eta 0:00:20
   ---- -----

In [1]:
import gradio as gr
import faiss
import pickle
from sentence_transformers import SentenceTransformer
from transformers import pipeline

### Load FAISS index + documents

In [ ]:
!pip uninstall fitz

In [9]:
!pip install frontend

In [2]:
import faiss
import os
import fitz  # PyMuPDF

In [3]:
index_path = r"C:\Users\ADMIN\Documents\rag_chatbot\faiss_index.bin"
faiss_index = faiss.read_index(index_path)
pdf_folder = r"C:\Users\ADMIN\Documents\rag_chatbot\pdfs"
documents = []

for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_folder, file)
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text("text")
        documents.append(text)

In [4]:
import json
index_path = r"C:\Users\ADMIN\Documents\rag_chatbot\faiss_index.bin"
faiss_index = faiss.read_index(index_path)
documents = []
with open(r"C:\Users\ADMIN\Documents\rag_chatbot\chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        documents.append(json.loads(line)["text"])

In [5]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

### Hugging Face LLM pipeline

In [6]:
generator = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cpu


In [7]:
def rag_chat(query, history=[]):
    # Embed query
    query_vec = embed_model.encode([query])
    
    # Search FAISS
    scores, idxs = faiss_index.search(query_vec, k=3)
    context = "\n".join([documents[i] for i in idxs[0]])
    
    # Build prompt
    prompt = f"Answer based only on the context below:\n\n{context}\n\nQuestion: {query}\nAnswer:"
    
    # Generate
    answer = generator(prompt, max_new_tokens=300, do_sample=True, temperature=0.3)[0]["generated_text"]
    
    # Clean output
    answer = answer.split("Answer:")[-1].strip()
    
    # Append to history
    history.append((query, answer))
    return history, history

### Gradio UI

In [8]:
with gr.Blocks() as demo:
    gr.Markdown("# RAG Chatbot with FAISS + Hugging Face")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Ask me about your PDFs...")
    clear = gr.Button("Clear Chat")

    def user_query(user_input, history):
        return rag_chat(user_input, history)

    msg.submit(user_query, [msg, chatbot], [chatbot, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

demo.launch(share=True)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_1960\2150621977.py:4: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7973b86b4dc905cb00.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
